In [1]:
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [4]:
annot_test = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_test.csv")

In [ ]:
annot_train = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv")
annot_train.head()

In [ ]:
annot_train[annot_train.image_id == '000434271f63a053c4128a0ba6352c7f']

In [ ]:
annot_train

In [ ]:
annot_train[annot_train.image_id == "0005e8e3701dfb1dd93d53e2ff537b6e"]

In [ ]:
annot_train[annot_train.class_name == "Pleural thickening"]


In [ ]:
annot_test.head()

In [ ]:
annot_test.shape

In [ ]:
annot_test.image_id.value_counts()

In [5]:
annot_test[annot_test.image_id == "ffccf1709d0081d122a1d1f9edbefdf1"]

,image_id,class_name,x_min,y_min,x_max,y_max
1750,ffccf1709d0081d122a1d1f9edbefdf1,Infiltration,1693.273214,862.133671,2361.349603,1785.430253
1751,ffccf1709d0081d122a1d1f9edbefdf1,Infiltration,544.782344,730.770336,1111.521303,1413.859678
1752,ffccf1709d0081d122a1d1f9edbefdf1,Pulmonary fibrosis,616.093868,790.822147,1040.209778,1252.470438
1753,ffccf1709d0081d122a1d1f9edbefdf1,Pulmonary fibrosis,1820.883311,918.432243,2293.791316,1537.716536


In [ ]:
annot_test[annot_test.image_id == "ab9dedb9ff4cd9e80dca74505b599105"].class_name.unique()

In [6]:
img_lbl_test = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/image_labels_test.csv")

In [ ]:
img_lbl_test.head()

In [8]:
img_lbl_test.columns

Index(['image_id', 'Aortic enlargement', 'Atelectasis', 'Calcification',
       'Cardiomegaly', 'Clavicle fracture', 'Consolidation', 'Edema',
       'Emphysema', 'Enlarged PA', 'ILD', 'Infiltration', 'Lung Opacity',
       'Lung cavity', 'Lung cyst', 'Mediastinal shift', 'Nodule/Mass',
       'Pleural effusion', 'Pleural thickening', 'Pneumothorax',
       'Pulmonary fibrosis', 'Rib fracture', 'Other lesion', 'COPD',
       'Lung tumor', 'Pneumonia', 'Tuberculosis', 'Other disease',
       'No finding'],
      dtype='object')

In [7]:
img_lbl_test[img_lbl_test.image_id == "ffccf1709d0081d122a1d1f9edbefdf1"].T

,612
image_id,ffccf1709d0081d122a1d1f9edbefdf1
Aortic enlargement,0
Atelectasis,0
Calcification,0
Cardiomegaly,0
Clavicle fracture,0
Consolidation,0
Edema,0
Emphysema,0
Enlarged PA,0


In [ ]:
images_root = "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr"
train_meta = pd.read_csv(Path(images_root) / "train_meta.csv")

In [ ]:
train_meta.head()

In [ ]:
x, y = train_meta.loc[train_meta['image_id'] == '4d390e07733ba06e5ff07412f09c0a92', 'dim0'].values, train_meta.loc[train_meta['image_id'] == '4d390e07733ba06e5ff07412f09c0a92', 'dim1'].values

In [ ]:
x, y

In [2]:
def load_meta(meta_csv):
    """
    Load VinDr meta CSV with columns: image_id, dim0, dim1
    Returns dict: image_id -> (orig_height, orig_width)
    """
    df = pd.read_csv(meta_csv)
    meta = {}
    for _, r in df.iterrows():
        image_id = r["image_id"]
        # dim0 = height, dim1 = width (typical numpy convention)
        orig_h = int(r["dim0"])
        orig_w = int(r["dim1"])
        meta[image_id] = (orig_h, orig_w)
    return meta

In [3]:
def visualize_vindr_boxes(
    image_id: str,
    images_root: str,
    annotations_csv: str,
    meta_csv: str,
    save_path: str = None,
):
    """
    Visualize a 256x256 VinDr PNG with its bounding boxes,
    scaling the original annotations using train_meta.csv.
    """
    # Load annotations
    df = pd.read_csv(annotations_csv)
    rows = df[df["image_id"] == image_id]
    if rows.empty:
        raise ValueError(f"No annotations found for image_id={image_id} in {annotations_csv}")

    # Load meta for original size
    meta = load_meta(meta_csv)
    if image_id not in meta:
        raise KeyError(f"image_id={image_id} not found in meta CSV {meta_csv}")
    orig_h, orig_w = meta[image_id]

    # Load 256x256 PNG
    img_path = Path(images_root) / f"{image_id}.png"
    if not img_path.exists():
        raise FileNotFoundError(f"PNG not found: {img_path}")
    img = Image.open(img_path).convert("RGB")
    w_png, h_png = img.size

    assert w_png == h_png == 256, f"Expected 256x256 PNG, got {w_png}x{h_png}"

    # Scaling factors from original -> 256x256
    scale_x = w_png / float(orig_w)
    scale_y = h_png / float(orig_h)

    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img, cmap="gray")
    ax.axis("off")

    for _, r in rows.iterrows():
        x_min_orig = r["x_min"]
        y_min_orig = r["y_min"]
        x_max_orig = r["x_max"]
        y_max_orig = r["y_max"]
        label = r.get("class_name", "")

        # Scale to PNG coordinates
        x_min = x_min_orig * scale_x
        y_min = y_min_orig * scale_y
        x_max = x_max_orig * scale_x
        y_max = y_max_orig * scale_y

        width = x_max - x_min
        height = y_max - y_min

        rect = patches.Rectangle(
            (x_min, y_min),
            width,
            height,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(rect)
        if label:
            ax.text(
                x_min,
                max(y_min - 2, 0),
                label,
                fontsize=8,
                color="yellow",
                bbox=dict(facecolor="black", alpha=0.5, edgecolor="none"),
            )

    plt.tight_layout()
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)
        print(f"Saved visualization to {save_path}")
    else:
        plt.show()

In [ ]:
visualize_vindr_boxes(
    "0005e8e3701dfb1dd93d53e2ff537b6e",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train_meta.csv",
)

In [ ]:
visualize_vindr_boxes(
    "0007d316f756b3fa0baea2ff514ce945",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train_meta.csv",
)

In [ ]:
img_lbl_test.columns[1:].values